# Part A · Reproduction with the authors' code (local RTX 3080)

Local twin of `colab_datasets.ipynb`: same sections, same configs, same calls into `original_code/`. Only what the hardware needs differs:

| Colab | Here |
| --- | --- |
| Drive mount, repo clone, `userdata` secrets | this checkout; `HF_TOKEN` (and optionally `DD2610_DATA`) in a git-ignored `.env` in the repo root |
| outputs symlinked to Drive (`MyDrive/asymmetric_duos/`) | outputs junctioned to `outputs/` in the repo root (git-ignored), plus `ray_results/` |
| `!python`, `!pip install` | `run(...)` with the `.venv` interpreter: on Windows `!` shows no output until the command exits. Packages installed once in the venv |
| `num_workers = os.cpu_count()` | capped at 8: Windows spawns each DataLoader worker as a fresh process |
| Ray pre-started with 4 logical CPUs | `hp_tune.py` starts its own Ray; this machine has enough CPUs |
| `transforms.Lambda(lambda x: x.convert("RGB"))` (fine with fork) | named `_to_rgb` in `original_code/load_models/transforms.py`: Windows spawns DataLoader workers and must pickle the transforms; same operation |
| Caltech-256 val loader restarts its workers every epoch | `persistent_workers=True` in `original_code/load_data/datasets.py`: starting 8 workers takes ~1 min on Windows; val has no randomness |
| no progress bars | `DD2610_PROGRESS=1` turns on a per-batch `tqdm_ray` bar in `original_code/train/hp_tune.py` (display only; off by default) |
| `train` → `val` symlink | directory junction (symlinks need admin on Windows) |

**Each session:** run sections 0–2, then the stage you need. Every stage skips work whose outputs already exist.

In [1]:
import os, sys, _winapi
from pathlib import Path
from dotenv import load_dotenv

REPO = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'original_code').is_dir())
load_dotenv(REPO / '.env')   # local counterpart of Colab's userdata secrets (git-ignored)
OUT = REPO / 'outputs'       # local counterpart of MyDrive/asymmetric_duos
PY = sys.executable          # the .venv interpreter runs the release's scripts
DATA = Path(os.environ.get('DD2610_DATA', Path.home() / 'dd2610-data')).resolve()
HF_TOKEN = os.environ.get('HF_TOKEN')   # only needed for ImageNet-1K
print('HF_TOKEN', 'set' if HF_TOKEN else 'missing')
print(REPO, DATA, PY, sep='\n')

HF_TOKEN set
c:\Users\Hugo\Documents\Hugo\dev\DD2610-Final-Project
C:\Users\Hugo\dd2610-data
c:\Users\Hugo\Documents\Hugo\dev\DD2610-Final-Project\.venv\Scripts\python.exe


## 0 · Code and environment
One-time setup in PowerShell from the repository root. Use Python 3.12: the release pins `numpy==1.26.4`, and PyPI's Windows torch wheels are CPU-only. pyarrow 23+ (imported by Ray Tune) needs NumPy 2, hence the pin. In VS Code, select `.venv` as the kernel.
```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126
.\.venv\Scripts\python.exe -m pip install -r original_code/requirements.txt fvcore huggingface_hub "pyarrow==22.0.0" python-dotenv ipykernel ipywidgets
```
Ray Tune's trial folders exceed Windows' 260-character path limit. Enable long paths once (admin PowerShell, then reboot):
`Set-ItemProperty HKLM:\SYSTEM\CurrentControlSet\Control\FileSystem LongPathsEnabled 1`

In [2]:
%cd {REPO / 'original_code'}
# The release writes to paths relative to original_code; junctions point them at outputs/.
for d in ['checkpoints', 'y-prediction', 'evaluation', 'configs', 'ray_results']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
    if not Path(d).exists():
        _winapi.CreateJunction(str(OUT / d), d)

# Scripts in train/ import the package folders, so they need original_code on the path.
sys.path.insert(0, os.getcwd())
os.environ.update(PYTHONPATH=os.getcwd(), WANDB_MODE='disabled', PYTHONUNBUFFERED='1',
                  DD2610_PROGRESS='1')   # per-batch tqdm bars in hp_tune.py

import subprocess

def run(cmd):
    """Run a release script with the venv's Python and stream its output live.
    Stands in for `!python`, whose output on Windows only appears when the command exits.
    Interrupting the cell kills the script and everything it started (including Ray)."""
    p = subprocess.Popen([PY, *cmd.split()], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        while chunk := p.stdout.read1(65536):
            sys.stdout.write(chunk.decode('utf-8', 'replace'))
            sys.stdout.flush()
    except KeyboardInterrupt:
        subprocess.run(['taskkill', '/F', '/T', '/PID', str(p.pid)], capture_output=True)
        raise
    if p.wait():
        raise RuntimeError(f'{cmd.split()[0]} exited with code {p.returncode}')

c:\Users\Hugo\Documents\Hugo\dev\DD2610-Final-Project\original_code


## 1 · Data
The authors' `ImageNetDataset` reads `ImageFolder(root/val)` and `ImageFolder(root/train)`. The ImageNet images are therefore written to class folders `000`…`999`, which sort into the standard label order. `train` points to `val` because we only evaluate pretrained ImageNet models.

The ImageNet-1K cell needs `HF_TOKEN` and access to the gated `ILSVRC/imagenet-1k` dataset; Caltech-256 training only needs the first cell.

In [3]:
from torchvision.datasets import Caltech256
from torchvision.datasets.utils import download_and_extract_archive

IMAGENET, IMAGENET_V2 = DATA / 'imagenet', DATA / 'imagenetv2-matched-frequency-format-val'

Caltech256(DATA, download=True)
if not IMAGENET_V2.exists():
    download_and_extract_archive('https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/'
                                 'imagenetv2-matched-frequency.tar.gz', DATA, filename='imagenetv2.tar', remove_finished=True)

In [ ]:
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq

if not (IMAGENET / 'train').exists():
    snapshot_download('ILSVRC/imagenet-1k', repo_type='dataset', local_dir=DATA / 'hf',
                      allow_patterns=['data/validation-*.parquet'], token=HF_TOKEN)
    for shard in sorted((DATA / 'hf/data').glob('validation-*.parquet')):
        for row in pq.read_table(shard).to_pylist():
            out = IMAGENET / 'val' / f"{row['label']:03d}" / Path(row['image']['path']).name
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(row['image']['bytes'])
    _winapi.CreateJunction(str(IMAGENET / 'val'), str(IMAGENET / 'train'))

In [4]:
# Check the seed-42 splits with the authors' loaders.
from load_data import get_dataloaders
from load_models.transforms import get_transforms

caltech = get_dataloaders('caltech256', DATA, 64, 2, get_transforms())
print({k: len(v.dataset) for k, v in caltech.items() if v})   # train 21424, val 4591, test 4592
if (IMAGENET / 'train').exists():
    imagenet = get_dataloaders('imagenet', IMAGENET, 64, 2, get_transforms(), ood_root_dir=IMAGENET_V2)
    print({k: len(v.dataset) for k, v in imagenet.items() if v})  # train 50000 (= val), val 2500, test 47500, ood_test 10000

{'train': 21424, 'val': 4591, 'test': 4592}


## 2 · Configs (missing from the release) and FLOPs
* **`backbones.csv`:** the 8 models from the README. `4_evaluate_predictions.py` also reads `GFLOPS` and `Params` from this file.
* **`duos.csv`:** 2 bases × 6 sidekicks with `gflops_*`. The FLOP count (step 8) happens here because `5_evaluate_duo_predictions.py` reads the GFLOPs from this file. We use fvcore on one 224×224 image with fused attention off. fvcore prints a warning for any operator it can't count.
* **`caltech256.yaml`:** batch size 128, 8 LP epochs and 16 FT epochs come from paper Appendix A.2. `warmup_epochs` and `grace_period` are **our choice**, because the paper doesn't give them. `num_workers` is a loader setting only; it does not change the results.

In [5]:
import pandas as pd, torch, timm, yaml
from fvcore.nn import FlopCountAnalysis
from load_models import get_model_with_head

timm.layers.set_fused_attn(False)
backbones = pd.DataFrame([
    ('convnext_base', 'torchvision', 'IMAGENET1K_V1'),
    ('vit_base_patch16_clip_224.laion2b_ft_in12k_in1k', 'timm', None),
    ('resnet10t.c3_in1k', 'timm', None),
    ('resnet18', 'torchvision', 'IMAGENET1K_V1'),   # ResNet weights: our choice
    ('resnet50', 'torchvision', 'IMAGENET1K_V1'),
    ('resnet101', 'torchvision', 'IMAGENET1K_V1'),
    ('resnet152', 'torchvision', 'IMAGENET1K_V1'),
    ('resnet200d.ra2_in1k', 'timm', None),
], columns=['model_name', 'source', 'tv_weights'])
BASES = backbones.model_name[:2].tolist()

for i, r in backbones.iterrows():
    model = get_model_with_head(r.model_name, None, r.source, r.tv_weights, keep_imagenet_head=True)[0].eval()
    backbones.loc[i, 'GFLOPS'] = FlopCountAnalysis(model, torch.randn(1, 3, 224, 224)).total() / 1e9
    backbones.loc[i, 'Params'] = sum(p.numel() for p in model.parameters()) / 1e6
backbones.to_csv('configs/backbones.csv', index=False)

g = backbones.set_index('model_name').GFLOPS
pd.DataFrame([(l, s, g[l], g[s], g[s] / g[l]) for l in BASES for s in g.index if s not in BASES],
             columns=['model_large', 'model_small', 'gflops_large', 'gflops_small', 'gflops_balance']
             ).to_csv('configs/duos.csv', index=False)
yaml.safe_dump({'training': {'batch_size': 128, 'num_epochs_lp': 8, 'num_epochs_ff': 16,
                             'num_workers': min(os.cpu_count(), 8), 'warmup_epochs': 2, 'grace_period': 4}},
               open('configs/caltech256.yaml', 'w'))
backbones

Unsupported operator aten::gelu encountered 36 time(s)
Unsupported operator aten::mul encountered 36 time(s)
Unsupported operator aten::add_ encountered 36 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
features.1.0.stochastic_depth, features.1.1.stochastic_depth, features.1.2.stochastic_depth, features.3.0.stochastic_depth, features.3.1.stochastic_depth, features.3.2.stochastic_depth, features.5.0.stochastic_depth, features.5.1.stochastic_depth, features.5.10.stochastic_depth, features.5.11.stochastic_depth, features.5.12.stochastic_depth, features.5.13.stochastic_depth, features.5.14.stochastic_depth, features.5.15.stochastic_depth, features.5.16.stochastic_depth, features.5.17.stochastic_depth, features.5

(timm) Model already has forward_features method, skipping addition, adding pooling.


Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::mul encountered 12 time(s)
Unsupported operator aten::softmax encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 4 time(s)
Unsupported operator aten::avg_pool2d encountered 3 time(s)


(timm) Model already has forward_features method, skipping addition, adding pooling.


Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 8 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 33 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 50 time(s)


(timm) Model already has forward_features method, skipping addition, adding pooling.


Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 66 time(s)
Unsupported operator aten::avg_pool2d encountered 3 time(s)


,model_name,source,tv_weights,GFLOPS,Params
0,convnext_base,torchvision,IMAGENET1K_V1,15.383636,88.591464
1,vit_base_patch16_clip_224.laion2b_ft_in12k_in1k,timm,NaN,17.583497,86.568424
2,resnet10t.c3_in1k,timm,NaN,1.102151,5.435488
3,resnet18,torchvision,IMAGENET1K_V1,1.819066,11.689512
4,resnet50,torchvision,IMAGENET1K_V1,4.111513,25.557032
5,resnet101,torchvision,IMAGENET1K_V1,7.833970,44.549160
6,resnet152,torchvision,IMAGENET1K_V1,11.558835,60.192808
7,resnet200d.ra2_in1k,timm,NaN,15.313204,64.693064


## 3 · ImageNet checkpoints  →  `train/download_imagenet_models.py`

In [ ]:
run('train/download_imagenet_models.py --backbone_csv configs/backbones.csv --model_save_dir checkpoints/imagenet')

## 4 · Caltech-256 LP → FT  →  `train/hp_tune.py`
The script requests 4 CPUs and 1 GPU per Ray trial, so with one GPU the trials run one after another. The script skips LP for a model whose LP checkpoint already exists.

**Time one small model first.** If the full sweep is over budget, choose a reduced protocol (for example fewer `FT_TRIALS`) *before* looking at any test results, and document it.

**Speed (resnet18, measured):** the first epoch of each trial spends ~2 min starting DataLoader workers; after that an epoch takes ~0.4 min of data loading.

**10 GB VRAM:** batch size 128 in fp32 fits ResNet-18. Measure peak memory before running larger models, which may not fit.

In [6]:
import time

FT_TRIALS = 12                      # authors' default
MODELS = ['resnet18']               # timing run first; later: backbones.model_name

for r in backbones[backbones.model_name.isin(MODELS)].itertuples():
    t0 = time.time()
    run(f'train/hp_tune.py --model_name {r.model_name} --source {r.source} --tv_weights {r.tv_weights} '
        f'--dataset_name caltech256 --dataset_dir {DATA} --num_samples {FT_TRIALS}')
    print(f'{r.model_name}: {(time.time() - t0) / 60:.1f} min')



======hp_tune_configs======
args=Namespace(model_name='resnet18', dataset_name='caltech256', dataset_dir='C:\\Users\\Hugo\\dd2610-data', config_dir='configs', source='torchvision', tv_weights='IMAGENET1K_V1', num_samples=12, skip_lp_if_exist=True)


os.getcwd()='c:\\Users\\Hugo\\Documents\\Hugo\\dev\\DD2610-Final-Project\\original_code'
lp checkpoint exists, skipping.
2026-11-22 01:39:06,405	WARNING authentication_token_setup.py:85 -- Token authentication is enabled for this Ray cluster. Set RAY_AUTH_MODE=disabled to opt out. For more information, see https://docs.ray.io/en/latest/ray-security/token-auth.html
2026-11-22 01:39:12,332	INFO worker.py:2038 -- Started a local Ray instance.
2026-11-22 01:39:16,693	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `tune.run(...)`.
2026-11-22 01:39:16,739	WARNING tune.py:900 -- AIR_VERBOSITY is set, ignoring passed-in ProgressReporter for now.
2026-11-22 01:39:16,7

## 5 · Logits  →  `evaluate/1_single_model_prediction.py`
Writes `y-prediction/<dataset>/{val,test,ood_test}/raw/<model>.csv`. Before any models are combined, the accuracy check confirms that each model's output indices match the dataset labels.

In [ ]:
run(f'evaluate/1_single_model_prediction.py --dataset_name imagenet --dataset_dir {IMAGENET} --ood_dataset_dir {IMAGENET_V2} '
    f'--model_dir_path checkpoints/imagenet --backbone_csv configs/backbones.csv')
run(f'evaluate/1_single_model_prediction.py --dataset_name caltech256 --dataset_dir {DATA} '
    f'--model_dir_path checkpoints/caltech256/ff --backbone_csv configs/backbones.csv')

for ds in ['imagenet', 'caltech256']:
    pp = pd.read_csv(f'y-prediction/{ds}/test/point_prediction.csv')
    print(ds, pp.drop(columns='target').eq(pp.target, axis=0).mean().round(3).to_dict())

## 6 · Temperature scaling  →  `evaluate/2_temp_scale.py`, `evaluate/3_duo_temp_scale.py`
Fits on the validation logits. `2_` fits one temperature per model, giving the temperature-scaled base. `3_` fits the two Duo weights, giving the learned-weight Duo. ImageNetV2 reuses the ImageNet values.

In [ ]:
for ds in ['imagenet', 'caltech256']:
    run(f'evaluate/2_temp_scale.py --dataset_name {ds} --prediction_dir_path y-prediction/{ds}/val/raw')
    run(f'evaluate/3_duo_temp_scale.py --dataset_name {ds} --prediction_dir_path y-prediction/{ds}/val/raw --duo_csv_path configs/duos.csv')

## 7 · Evaluation  →  `evaluate/4_evaluate_predictions.py`, `evaluate/5_evaluate_duo_predictions.py`
How `mode` in the output maps to our comparisons:
* `logit_average` → equal-weight Duo
* `temperature_weighted` → learned-weight Duo
* `dictatorial_weighteduncertainty` → UQ-only

The temperature-scaled base comes from `4_`. Metrics: accuracy, AUROC, AURC, SAC@98.

In [ ]:
for ds in ['imagenet', 'caltech256']:
    run(f'evaluate/4_evaluate_predictions.py --dataset_name {ds} --backbone_csv configs/backbones.csv --prediction_dir_path y-prediction/{ds}')
    run(f'evaluate/5_evaluate_duo_predictions.py --dataset_name {ds} --duo_csv_path configs/duos.csv --prediction_dir_path y-prediction/{ds}')

## 8 · FLOP balance
Computed in step 2: `gflops_balance` = sidekick GFLOPs ÷ base GFLOPs in `configs/duos.csv`.

## 9 · Evaluator discrepancy (to do)
In `evaluate/5_evaluate_duo_predictions.py` (around lines 111 and 128), the `temperature_weighted` Duo reuses the UQ-only confidence, which is the Duo probability at the *base* model's prediction. Paper §3.1 scores the Duo's confidence on its *own* predicted class. We report both, with the paper-definition results clearly labelled.

## 10 · Figures (to do)
Plot the metrics against `gflops_balance` from `evaluation/eval_res/` to reproduce Figures 3–6, starting from `plot_notebooks/plot_2_duos.ipynb`. Compare Caltech-256 with Appendix G Figures 11 and 14.